# TCGA Wet-Lab Validated Target Mining — Pipeline Analysis

This notebook analyzes the output of the full 33-cancer pipeline run.
It **reads only** from `../output/` and `../data/` — no project code is modified.

**Run:** full 33-cancer pipeline  
**Data source:** `../output/final_targets.csv` + `../data/extractions_all.json`

## 1. Setup & Data Loading

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.figsize': (10, 6),
})
sns.set_style('whitegrid')

# Color palette for 33 cancer types
CANCER_PALETTE = sns.color_palette('husl', 33)

# Load data
df = pd.read_csv('../output/final_targets.csv')
with open('../data/extractions_all.json') as f:
    extractions = json.load(f)

print(f'Final targets CSV: {len(df):,} rows, {df["tcga_code"].nunique()} cancer types')
print(f'Extractions JSON: {len(extractions)} cancer types')

In [ ]:
# Build per-cancer paper-level stats from extractions JSON
cancer_stats = []
for code, papers in extractions.items():
    n_total = len(papers)
    n_wet = sum(1 for p in papers if p.get('has_wet_lab_validation'))
    n_review = sum(1 for p in papers if p.get('is_review'))
    n_insufficient = sum(1 for p in papers if p.get('insufficient_info'))
    n_other = n_total - n_wet - n_review - n_insufficient
    n_targets = sum(len(p.get('validated_targets', [])) for p in papers if p.get('has_wet_lab_validation'))
    cancer_stats.append({
        'tcga_code': code,
        'screened': n_total,
        'wet_lab': n_wet,
        'review': n_review,
        'insufficient': n_insufficient,
        'other': n_other,
        'raw_targets': n_targets,
        'wet_rate': n_wet / max(n_total, 1) * 100,
    })
stats_df = pd.DataFrame(cancer_stats)

# Merge with final dedup target counts
dedup_counts = df.groupby('tcga_code').size().reset_index(name='final_targets')
stats_df = stats_df.merge(dedup_counts, on='tcga_code', how='left')
stats_df['final_targets'] = stats_df['final_targets'].fillna(0).astype(int)

print(f'Per-cancer stats: {len(stats_df)} cancer types')
stats_df.head()

## 2. Pipeline Overview

In [ ]:
total_screened = stats_df['screened'].sum()
total_wet = stats_df['wet_lab'].sum()
total_targets = len(df)
unique_targets = df['target'].nunique()
cancers_with_targets = df['tcga_code'].nunique()

# Gene standardization
gene_mask = df['official_symbol'].notna() & (df['official_symbol'] != '')
mapped_genes = gene_mask.sum()

print('=' * 55)
print('  TCGA Wet-Lab Validated Target Mining — Pipeline Results')
print('=' * 55)
print(f'  Cancer types:                  {len(stats_df):>4}')
print(f'  Papers screened:               {total_screened:>6,}')
print(f'  Wet-lab validated papers:      {total_wet:>6,} ({total_wet/total_screened*100:.1f}%)')
print(f'  Raw target mentions (pre-dedup): {stats_df["raw_targets"].sum():>6,}')
print(f'  Final target-disease pairs:    {total_targets:>6,}')
print(f'  Unique targets:                {unique_targets:>6,}')
print(f'  Cancer types with targets:     {cancers_with_targets:>4}')
print(f'  Gene targets standardized:     {mapped_genes:>6,}')
print(f'  Cross-cancer (>=3):            {(df.groupby("target")["tcga_code"].nunique() >= 3).sum():>6,}')
print('=' * 55)

In [ ]:
# Four-category pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = ['wet_lab', 'review', 'insufficient', 'other']
labels = [f'Wet Lab\n({total_wet:,})', f'Review/Bioinfo\n({stats_df["review"].sum():,})',
          f'Insufficient Info\n({stats_df["insufficient"].sum():,})', f'Other\n({stats_df["other"].sum():,})']
sizes = [stats_df[c].sum() for c in categories]
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6']

wedges, texts, autotexts = axes[0].pie(sizes, labels=labels, colors=colors,
                                         autopct='%1.1f%%', startangle=90,
                                         explode=(0.05, 0, 0, 0))
for at in autotexts:
    at.set_fontsize(10)
axes[0].set_title('Paper Classification')

# Wet lab rate per cancer histogram
axes[1].hist(stats_df['wet_rate'], bins=15, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].axvline(stats_df['wet_rate'].mean(), color='red', linestyle='--',
                label=f'Mean: {stats_df["wet_rate"].mean():.1f}%')
axes[1].set_xlabel('Wet-Lab Rate (%)')
axes[1].set_ylabel('Number of Cancer Types')
axes[1].set_title('Distribution of Wet-Lab Rate Across 33 Cancers')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Per-Cancer Landscape

In [ ]:
# Sort by final targets
sorted_stats = stats_df.sort_values('final_targets', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 14))

# Left: final targets per cancer
bars = axes[0].barh(range(len(sorted_stats)), sorted_stats['final_targets'],
                     color=['#3498db' if v > 0 else '#ecf0f1' for v in sorted_stats['final_targets']])
axes[0].set_yticks(range(len(sorted_stats)))
axes[0].set_yticklabels(sorted_stats['tcga_code'], fontsize=8, fontfamily='monospace')
axes[0].set_xlabel('Final Target-Disease Pairs')
axes[0].set_title('Targets per Cancer (after dedup)')
for i, (v, code) in enumerate(zip(sorted_stats['final_targets'], sorted_stats['tcga_code'])):
    if v > 0:
        axes[0].text(v + 5, i, str(v), va='center', fontsize=8)

# Right: wet lab papers per cancer
axes[1].barh(range(len(sorted_stats)), sorted_stats['wet_lab'],
             color=['#2ecc71' if v > 0 else '#ecf0f1' for v in sorted_stats['wet_lab']])
axes[1].set_yticks(range(len(sorted_stats)))
axes[1].set_yticklabels(sorted_stats['tcga_code'], fontsize=8, fontfamily='monospace')
axes[1].set_xlabel('Wet-Lab Papers')
axes[1].set_title('Wet-Lab Papers per Cancer')
for i, (v, code) in enumerate(zip(sorted_stats['wet_lab'], sorted_stats['tcga_code'])):
    if v > 0:
        axes[1].text(v + 1, i, str(v), va='center', fontsize=8)

plt.tight_layout()
plt.savefig('figures/per_cancer.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top / bottom 5 cancers
top5 = sorted_stats.nlargest(5, 'final_targets')[['tcga_code', 'screened', 'wet_lab', 'final_targets', 'wet_rate']]
bottom5 = sorted_stats[lambda x: x['final_targets'] > 0].nsmallest(5, 'final_targets')[['tcga_code', 'screened', 'wet_lab', 'final_targets', 'wet_rate']]
empty = sorted_stats[lambda x: x['final_targets'] == 0]

print('Top 5 cancers by target count:')
display(top5.style.background_gradient(subset=['final_targets'], cmap='Greens'))
print(f'\nBottom 5 (non-zero):')
display(bottom5)
if len(empty) > 0:
    print(f'\n{len(empty)} cancer(s) with ZERO targets: {", ".join(empty["tcga_code"])}')
else:
    print('\nAll 33 cancers have at least 1 target ✓')

## 4. Target Type Distribution

In [ ]:
type_counts = df['target_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie
top_types = type_counts.head(6)
other_count = type_counts[6:].sum()
pie_data = list(top_types.items()) + ([('other', other_count)] if other_count > 0 else [])
pie_colors = sns.color_palette('Set2', len(pie_data))
axes[0].pie([v for _, v in pie_data], labels=[f'{k}\n({v:,})' for k, v in pie_data],
            colors=pie_colors, autopct='%1.1f%%', startangle=140)
axes[0].set_title('Target Type Distribution')

# Bar
bars = axes[1].bar(type_counts.index, type_counts.values, color=sns.color_palette('Set2', len(type_counts)))
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Target Type')
axes[1].set_title('Target Type Counts')
axes[1].tick_params(axis='x', rotation=45)
for bar, v in zip(bars, type_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(v),
                 ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/target_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Top Targets & Cross-Cancer Analysis

In [ ]:
# Cross-cancer targets
cross_cancer = df.groupby('target')['tcga_code'].nunique().sort_values(ascending=False)
multi_cancer = cross_cancer[cross_cancer >= 3]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 25 cross-cancer targets
top25 = cross_cancer.head(25)
colors = ['#e74c3c' if v >= 10 else '#f39c12' if v >= 5 else '#3498db' for v in top25.values]
axes[0].barh(range(len(top25)), top25.values, color=colors)
axes[0].set_yticks(range(len(top25)))
axes[0].set_yticklabels(top25.index, fontfamily='monospace', fontsize=9)
axes[0].set_xlabel('Number of Cancer Types')
axes[0].set_title(f'Top 25 Cross-Cancer Targets\n({len(multi_cancer)} targets appear in >=3 cancers)')
axes[0].invert_yaxis()
for i, v in enumerate(top25.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=8)

# Histogram: how many cancers does each target appear in?
axes[1].hist(cross_cancer.values, bins=range(1, int(cross_cancer.max()) + 2),
             color='#3498db', edgecolor='white', alpha=0.8)
axes[1].axvline(3, color='red', linestyle='--', label='Threshold (>=3)')
axes[1].set_xlabel('Number of Cancer Types')
axes[1].set_ylabel('Number of Targets')
axes[1].set_title('Distribution: Targets per Cancer Type Count')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/cross_cancer.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Targets appearing in 1 cancer:    {(cross_cancer == 1).sum():,}')
print(f'Targets appearing in 2 cancers:   {(cross_cancer == 2).sum():,}')
print(f'Targets appearing in 3+ cancers:  {(cross_cancer >= 3).sum():,}')
print(f'Targets appearing in 5+ cancers:  {(cross_cancer >= 5).sum():,}')
print(f'Targets appearing in 10+ cancers: {(cross_cancer >= 10).sum():,}')

In [ ]:
# Cross-cancer targets table (>=5 cancers)
top_cross = cross_cancer[cross_cancer >= 5].reset_index()
top_cross.columns = ['Target', 'Cancer Types']
top_cross['Cancers'] = top_cross['Target'].apply(
    lambda x: ', '.join(sorted(df[df['target'] == x]['tcga_code'].unique()))
)
display(top_cross.style.background_gradient(subset=['Cancer Types'], cmap='Oranges'))

## 6. Functional Role Distribution

In [ ]:
# Clean functional_role: extract base role (before majority-vote annotation)
def clean_role(role):
    if pd.isna(role) or role == 'Null':
        return 'Null'
    # Remove majority vote annotation like ' (3/5)'
    return str(role).split(' (')[0]

df['role_clean'] = df['functional_role'].apply(clean_role)
role_counts = df['role_clean'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

role_colors = {
    'Oncogene': '#e74c3c',
    'Tumor suppressor': '#2ecc71',
    'Biomarker': '#3498db',
    'Protective': '#f1c40f',
    'Risk': '#e67e22',
    'Null': '#95a5a6',
}
colors = [role_colors.get(r, '#95a5a6') for r in role_counts.index]

axes[0].pie(role_counts.values, labels=[f'{k}\n({v:,})' for k, v in role_counts.items()],
            colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Functional Role Distribution')

# Non-null rate per cancer
non_null_rate = df.groupby('tcga_code')['role_clean'].apply(
    lambda x: (x != 'Null').sum() / len(x) * 100
).sort_values()

axes[1].barh(range(len(non_null_rate)), non_null_rate.values,
             color=['#2ecc71' if v > 50 else '#f39c12' if v > 20 else '#e74c3c' for v in non_null_rate.values])
axes[1].set_yticks(range(len(non_null_rate)))
axes[1].set_yticklabels(non_null_rate.index, fontsize=8, fontfamily='monospace')
axes[1].set_xlabel('% with Functional Role')
axes[1].set_title('Functional Role Annotation Rate per Cancer')

plt.tight_layout()
plt.savefig('figures/functional_role.png', dpi=150, bbox_inches='tight')
plt.show()

overall_rate = (df['role_clean'] != 'Null').sum() / len(df) * 100
print(f'Overall functional role annotation rate: {overall_rate:.1f}%')
print(f'Targets with role = Null: {(df["role_clean"] == "Null").sum():,} / {len(df):,}')

## 7. Expression Change Patterns

In [ ]:
expr_counts = df['expression_change'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
expr_colors = {'Upregulated': '#e74c3c', 'Downregulated': '#3498db', 'Unchanged': '#95a5a6', 'Null': '#bdc3c7'}
colors = [expr_colors.get(k, '#95a5a6') for k in expr_counts.index]
bars = ax.bar(expr_counts.index, expr_counts.values, color=colors, edgecolor='white')
for bar, v in zip(bars, expr_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{v:,}', ha='center', fontsize=11)
ax.set_ylabel('Count')
ax.set_title('Expression Change Distribution')
plt.tight_layout()
plt.savefig('figures/expression.png', dpi=150, bbox_inches='tight')
plt.show()

non_null_expr = (df['expression_change'] != 'Null').sum() / len(df) * 100
print(f'Targets with expression change annotated: {non_null_expr:.1f}%')

## 8. Literature Characteristics

In [ ]:
# Top journals
journal_counts = df['journal'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(range(len(journal_counts)), journal_counts.values, color='#3498db')
axes[0].set_yticks(range(len(journal_counts)))
axes[0].set_yticklabels(journal_counts.index, fontsize=9)
axes[0].set_xlabel('Target-Disease Pairs')
axes[0].set_title('Top 15 Journals')
axes[0].invert_yaxis()
for i, v in enumerate(journal_counts.values):
    axes[0].text(v + 1, i, str(v), va='center', fontsize=8)

# Year distribution: parse start year from ranges like "2005-2025"
def parse_start_year(y):
    try:
        return int(str(y).split('-')[0])
    except:
        return None

df['year_start'] = df['year'].apply(parse_start_year)
year_counts = df['year_start'].value_counts().sort_index()
axes[1].bar(year_counts.index, year_counts.values, color='#2ecc71', edgecolor='white')
axes[1].set_xlabel('Publication Year (start)')
axes[1].set_ylabel('Target-Disease Pairs')
axes[1].set_title('Year Distribution')
axes[1].axvline(df['year_start'].median(), color='red', linestyle='--',
                label=f'Median: {df["year_start"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/literature.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Year range: {df["year_start"].min():.0f} - {df["year_start"].max():.0f}')
print(f'Median year: {df["year_start"].median():.0f}')

In [ ]:
# Top validation methods
all_methods = []
for methods_str in df['validation_methods'].dropna():
    for m in str(methods_str).split(';'):
        m = m.strip().strip(',').strip()
        if m:
            all_methods.append(m)

method_counts = Counter(all_methods).most_common(20)

fig, ax = plt.subplots(figsize=(10, 6))
methods, counts = zip(*method_counts)
ax.barh(range(len(methods)), counts, color='#9b59b6')
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods, fontsize=9)
ax.set_xlabel('Occurrences')
ax.set_title('Top 20 Validation Methods')
ax.invert_yaxis()
for i, v in enumerate(counts):
    ax.text(v + 1, i, str(v), va='center', fontsize=8)
plt.tight_layout()
plt.savefig('figures/methods.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Quality Checks

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Supporting papers distribution
axes[0, 0].hist(df['n_papers'], bins=range(1, df['n_papers'].max() + 2),
                color='#3498db', edgecolor='white', alpha=0.8)
axes[0, 0].set_xlabel('Number of Supporting Papers')
axes[0, 0].set_ylabel('Target-Disease Pairs')
axes[0, 0].set_title('Multi-Paper Support Distribution')
axes[0, 0].axvline(df['n_papers'].median(), color='red', linestyle='--', label=f'Median: {df["n_papers"].median():.0f}')
axes[0, 0].legend()

single = (df['n_papers'] == 1).sum()
multi = (df['n_papers'] >= 2).sum()
high = (df['n_papers'] >= 5).sum()

# Gene standardization by type
type_gene_map = df.groupby('target_type').apply(
    lambda g: (g['official_symbol'].notna() & (g['official_symbol'] != '')).sum() / len(g) * 100
).sort_values(ascending=False)
type_gene_map = type_gene_map[type_gene_map > 0]
axes[0, 1].barh(range(len(type_gene_map)), type_gene_map.values, color='#2ecc71')
axes[0, 1].set_yticks(range(len(type_gene_map)))
axes[0, 1].set_yticklabels(type_gene_map.index, fontsize=9)
axes[0, 1].set_xlabel('Standardization Rate (%)')
axes[0, 1].set_title('Gene Standardization Rate by Target Type')
for i, v in enumerate(type_gene_map.values):
    axes[0, 1].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)

# n_papers vs functional role
has_role = df['role_clean'] != 'Null'
axes[1, 0].boxplot([df[has_role]['n_papers'], df[~has_role]['n_papers']],
                   labels=['Has Functional Role', 'Role = Null'], patch_artist=True)
axes[1, 0].set_ylabel('Number of Supporting Papers')
axes[1, 0].set_title('Paper Support vs Functional Role Annotation')

# Model type distribution
model_counts = df['model_type'].value_counts().head(8)
axes[1, 1].barh(range(len(model_counts)), model_counts.values, color='#e67e22')
axes[1, 1].set_yticks(range(len(model_counts)))
axes[1, 1].set_yticklabels(model_counts.index, fontsize=9)
axes[1, 1].set_xlabel('Target-Disease Pairs')
axes[1, 1].set_title('Top Model Types')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig('figures/quality.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Single-paper targets:  {single:,} ({single/len(df)*100:.1f}%)')
print(f'Multi-paper (>=2):     {multi:,} ({multi/len(df)*100:.1f}%)')
print(f'Highly supported (>=5): {high:,} ({high/len(df)*100:.1f}%)')

## 10. Summary

In [ ]:
print('=' * 60)
print('  ANALYSIS COMPLETE')
print('=' * 60)
print(f'  Cancer types with targets:    {cancers_with_targets}/33')
print(f'  Total target-disease pairs:   {total_targets:,}')
print(f'  Unique targets:               {unique_targets:,}')
print(f'  Wet-lab validation rate:      {total_wet/total_screened*100:.1f}%')
print(f'  Gene standardization rate:    {mapped_genes/len(df)*100:.1f}%')
print(f'  Functional role annotated:    {overall_rate:.1f}%')
print(f'  Multi-paper support (>=2):    {multi/len(df)*100:.1f}%')
print(f'  Cross-cancer targets (>=3):   {len(multi_cancer):,}')
print(f'')
print(f'  Figures saved to: analysis/figures/')
print(f'  - overview.png')
print(f'  - per_cancer.png')
print(f'  - target_types.png')
print(f'  - cross_cancer.png')
print(f'  - functional_role.png')
print(f'  - expression.png')
print(f'  - literature.png')
print(f'  - methods.png')
print(f'  - quality.png')
print('=' * 60)